# Notebook 4: nnU-Net Official — MSD Task03 Liver (2D Config)

Uses the official `nnunetv2` library. All training runs via `subprocess` so the kernel keeps the process alive even if you close the browser — safe for overnight/multi-day runs.

| | |
|---|---|
| Library | `nnunetv2` (official) |
| Config | `2d` |
| Expected tumor Dice | 0.74–0.76 (fold 0), 0.75–0.78 (5-fold mean) |
| Baseline to beat | 0.7139 |
| Disk space needed | ~5–8 GB (2D config only, no `--npz`) |

**Prereq:** `nnunetv2` must already be installed in the `liver2` env:
```bash
conda activate liver2
pip install nnunetv2
```

---
### How to leave this running safely overnight
Training cells use `subprocess` — attached to the **kernel**, not the browser tab. Training continues even if you close the browser, your laptop sleeps, or the SSH session drops.

> ⚠️ Only edit **Cell 2 (Paths)**. Everything else runs as-is.

## 1. Imports

In [5]:
import os
import sys
import json
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch : 2.10.0+cu126
CUDA    : True
GPU     : NVIDIA GeForce RTX 4060
VRAM    : 8.6 GB


## 2. Paths — Only Edit This Cell

In [6]:
# ── Edit these two lines only ─────────────────────────────────────────────────
TASK03_SRC  = r"C:\Users\DYPIU\Desktop\08-3D-Liver-Tumor-Segmentation\data\Task03_Liver"
NNUNET_BASE = r"C:\Users\DYPIU\Desktop\08-3D-Liver-Tumor-Segmentation\nnunet_data"    # where nnU-Net writes everything (~5-8 GB needed)
# ─────────────────────────────────────────────────────────────────────────────

NNUNET_RAW     = os.path.join(NNUNET_BASE, "nnUNet_raw")
NNUNET_PREP    = os.path.join(NNUNET_BASE, "nnUNet_preprocessed")
NNUNET_RESULTS = os.path.join(NNUNET_BASE, "nnUNet_results")
DATASET_ID     = "003"
DATASET_NAME   = f"Dataset{DATASET_ID}_Liver"
TRAINER = "nnUNetTrainer_500epochs"

for d in [NNUNET_RAW, NNUNET_PREP, NNUNET_RESULTS]:
    os.makedirs(d, exist_ok=True)

# Set env vars — required by every nnUNetv2 command
os.environ["nnUNet_raw"]          = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREP
os.environ["nnUNet_results"]      = NNUNET_RESULTS

print("nnUNet_raw         :", NNUNET_RAW)
print("nnUNet_preprocessed:", NNUNET_PREP)
print("nnUNet_results     :", NNUNET_RESULTS)
print()

# Check disk space
total, used, free = shutil.disk_usage(os.path.dirname(NNUNET_BASE))
print(f"Free disk space : {free / 1e9:.1f} GB  (need ~5-8 GB)")
if free / 1e9 < 8:
    print("⚠️  Less than 8 GB free — consider a different NNUNET_BASE path")

nnUNet_raw         : C:\Users\DYPIU\Desktop\08-3D-Liver-Tumor-Segmentation\nnunet_data\nnUNet_raw
nnUNet_preprocessed: C:\Users\DYPIU\Desktop\08-3D-Liver-Tumor-Segmentation\nnunet_data\nnUNet_preprocessed
nnUNet_results     : C:\Users\DYPIU\Desktop\08-3D-Liver-Tumor-Segmentation\nnunet_data\nnUNet_results

Free disk space : 27.9 GB  (need ~5-8 GB)


## 3. Convert MSD Dataset

nnU-Net v2 has a built-in MSD converter — no manual file renaming or `dataset.json` editing needed.
It handles the `_0000` suffix and `channel_names` key automatically.

In [7]:
dst = Path(NNUNET_RAW) / DATASET_NAME

if dst.exists():
    print(f"✓ Dataset already converted at {dst}")
    print(f"  imagesTr : {len(list((dst/'imagesTr').glob('*.nii.gz')))} files")
    print(f"  labelsTr : {len(list((dst/'labelsTr').glob('*.nii.gz')))} files")
else:
    print(f"Converting {TASK03_SRC} → {dst} ...")
    !nnUNetv2_convert_MSD_dataset -i "{TASK03_SRC}" -overwrite_id {DATASET_ID}

# Verify dataset.json
with open(dst / "dataset.json") as f:
    meta = json.load(f)
print("\ndataset.json:")
for k in ["channel_names", "labels", "numTraining", "file_ending"]:
    if k in meta:
        print(f"  {k}: {meta[k]}")

✓ Dataset already converted at C:\Users\DYPIU\Desktop\08-3D-Liver-Tumor-Segmentation\nnunet_data\nnUNet_raw\Dataset003_Liver
  imagesTr : 131 files
  labelsTr : 131 files

dataset.json:
  channel_names: {'0': 'CT'}
  labels: {'background': 0, 'liver': 1, 'cancer': 2}
  numTraining: 131
  file_ending: .nii.gz


## 4. Fingerprint, Plan & Preprocess

nnU-Net auto-configures patch size, batch size, spacing, and normalisation from the dataset.
`-c 2d` preprocesses the 2D config only — saves time and disk space.


In [ ]:
## 4. Fingerprint, Plan & Preprocess

# Verify env vars still set (re-run Cell 2 if you restarted the kernel)
for var in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    val = os.environ.get(var, "NOT SET")
    status = "✓" if val != "NOT SET" else "✗  — re-run Cell 2 first"
    print(f"  {status}  {var}")

prep_folder = Path(NNUNET_PREP) / DATASET_NAME / "nnUNetPlans_2d"
b2nd_files  = list(prep_folder.glob("*.b2nd"))
required    = 262   # 131 image + 131 seg .b2nd files

if len(b2nd_files) >= required:
    print(f"\n✅ Preprocessing already complete ({len(b2nd_files)} .b2nd files found)")
    print("   Skipping — no files will be overwritten")
else:
    missing = required - len(b2nd_files)
    print(f"\n⚠️  Only {len(b2nd_files)}/{required} .b2nd files found ({missing} missing)")
    print("   Running preprocessing — estimated 3–4 h on single process...")

    result = subprocess.run(
        ["nnUNetv2_plan_and_preprocess", "-d", DATASET_ID, "-c", "2d", "-np", "1", "-npfp", "1"],
        env=os.environ,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-4000:] if result.stdout else "")
    if result.stderr:
        print("STDERR:", result.stderr[-2000:])

    # Verify after completion
    b2nd_files = list(prep_folder.glob("*.b2nd"))
    if len(b2nd_files) >= required:
        print(f"\n✅ Preprocessing complete ({len(b2nd_files)} .b2nd files)")
    else:
        print(f"\n❌ Still missing files ({len(b2nd_files)}/{required}) — re-run this cell")

# Print what nnU-Net auto-configured for 2D
plan_path = Path(NNUNET_PREP) / DATASET_NAME / "nnUNetPlans.json"

if plan_path.exists():
    with open(plan_path) as f:
        plan = json.load(f)
    cfg = plan["configurations"]["2d"]
    print("\nnnU-Net 2D auto-configuration:")
    print(f"  Patch size     : {cfg.get('patch_size')}")
    print(f"  Batch size     : {cfg.get('batch_size')}")
    print(f"  Target spacing : {cfg.get('spacing')}")
    print(f"  Norm scheme    : {cfg.get('normalization_schemes')}")
else:
    print("\nPlan not found — preprocessing may still be running or failed")

  ✓  nnUNet_raw
  ✓  nnUNet_preprocessed
  ✓  nnUNet_results

⚠️  Only 4/262 .b2nd files found (258 missing)
   Running preprocessing — estimated 3–4 h on single process...


## 5. Train — All 5 Folds (Overnight)

Uses `subprocess` so training keeps running even if you close the browser.
Each fold trains for 1000 epochs × 250 iters. nnU-Net checkpoints every 50 epochs automatically.

| | |
|---|---|
| Each fold | ~2–4 h on RTX 4060 |
| All 5 folds | ~10–20 h |
| Resume interrupted fold | re-run same cell — nnU-Net detects checkpoint and continues |



In [ ]:
FOLDS = [0, 1, 2, 3, 4]

for fold in FOLDS:
    print(f"\n{'='*55}")
    print(f" Starting fold {fold}  ({time.strftime('%Y-%m-%d %H:%M:%S')})")
    print(f"{'='*55}")

    # Limit data loading workers to avoid RAM OOM on 16 GB system
    train_env = os.environ.copy()
    train_env["nnUNet_n_proc_DA"] = "2"   # reduce augmentation workers from default 12

    result = subprocess.run(
        ["nnUNetv2_train", DATASET_ID, "2d", str(fold),
         "-tr", TRAINER],
        env=train_env,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-4000:] if result.stdout else "")
    if result.stderr:
        print("STDERR:", result.stderr[-1000:])

    if result.returncode == 0:
        print(f"\n✅ Fold {fold} complete  ({time.strftime('%Y-%m-%d %H:%M:%S')})")
    else:
        print(f"\n❌ Fold {fold} failed (return code {result.returncode})")
        print("   Re-run this cell — nnU-Net resumes from last checkpoint")
        break

print(f"\n🎉 All folds done  ({time.strftime('%Y-%m-%d %H:%M:%S')})")

## 6. Evaluate — Validation Dice Per Fold

In [ ]:
for fold in FOLDS:
    print(f"Evaluating fold {fold}...")
    result = subprocess.run(
        ["nnUNetv2_train", DATASET_ID, "2d", str(fold),
         "-tr", TRAINER, "--val"],
        env=os.environ,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout[-2000:])

## 7. Results — Dice vs Baseline

In [ ]:
model_dir = Path(NNUNET_RESULTS) / DATASET_NAME / f"{TRAINER}__nnUNetPlans__2d"

liver_dices, tumor_dices = [], []

for fold in range(5):
    sp = model_dir / f"fold_{fold}" / "validation" / "summary.json"
    if not sp.exists():
        print(f"  Fold {fold}: not found — skipping")
        continue
    with open(sp) as f:
        s = json.load(f)
    per_class = s.get("mean", {})
    liver = per_class.get("1", {}).get("Dice", None)
    tumor = per_class.get("2", {}).get("Dice", None)
    if liver and tumor:
        liver_dices.append(liver)
        tumor_dices.append(tumor)
        print(f"  Fold {fold}:  Liver={liver:.4f}  Tumor={tumor:.4f}")

if liver_dices:
    print()
    print("="*47)
    print(" nnU-Net 2D Results")
    print("="*47)
    print(f" {'Metric':<12} {'Baseline':>10} {'nnU-Net 2D':>12}")
    print(f" {'-'*36}")
    print(f" {'Liver Dice':<12} {'0.8327':>10} {np.mean(liver_dices):>12.4f} ± {np.std(liver_dices):.4f}")
    print(f" {'Tumor Dice':<12} {'0.7139':>10} {np.mean(tumor_dices):>12.4f} ± {np.std(tumor_dices):.4f}")
    print(f" {'-'*36}")
    beat = "✅ BEAT baseline" if np.mean(tumor_dices) > 0.7139 else "❌ Below baseline"
    print(f" {beat}")
    print("="*47)
else:
    print("No results found yet — run Cell 6 first")

## 8. Training Curves

In [ ]:
model_dir = Path(NNUNET_RESULTS) / DATASET_NAME / f"{TRAINER}__nnUNetPlans__2d"

fig, axes = plt.subplots(1, len(FOLDS), figsize=(5 * len(FOLDS), 4))
if len(FOLDS) == 1:
    axes = [axes]

for ax, fold in zip(axes, FOLDS):
    log_path = model_dir / f"fold_{fold}" / "training_log.txt"
    if not log_path.exists():
        ax.set_title(f"Fold {fold} — no log yet")
        continue

    train_loss, val_pseudo_dice = [], []
    with open(log_path) as f:
        for line in f:
            if "train_loss" in line:
                try: train_loss.append(float(line.strip().split()[-1]))
                except: pass
            if "pseudo dice" in line.lower() or "ema_fg_dice" in line:
                try: val_pseudo_dice.append(float(line.strip().split()[-1]))
                except: pass

    if train_loss:
        ax.plot(train_loss, color="steelblue", label="Train loss")
        ax.set_ylabel("Loss", color="steelblue")
    if val_pseudo_dice:
        ax2 = ax.twinx()
        ax2.plot(val_pseudo_dice, color="crimson", label="Val pseudo-Dice")
        ax2.axhline(0.7139, color="gray", linestyle="--", linewidth=0.8, label="Baseline 0.7139")
        ax2.set_ylabel("Pseudo-Dice", color="crimson")

    ax.set_title(f"Fold {fold}")
    ax.set_xlabel("Epoch")

plt.suptitle("nnU-Net 2D — Training Progress", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("nnunet_training_curves.png", dpi=150)
plt.show()

## 9. Inference on Test Set

In [ ]:
TEST_INPUT  = str(Path(NNUNET_RAW) / DATASET_NAME / "imagesTs")
TEST_OUTPUT = str(Path(NNUNET_RESULTS) / "predictions_2d")
INFER_FOLD  = 0   # change to whichever fold had best Dice

os.makedirs(TEST_OUTPUT, exist_ok=True)

result = subprocess.run([
    "nnUNetv2_predict",
    "-i", TEST_INPUT,
    "-o", TEST_OUTPUT,
    "-d", DATASET_ID,
    "-c", "2d",
    "-f", str(INFER_FOLD),
    "-tr", TRAINER,
], env=os.environ, capture_output=True, text=True)
if result.stdout: print(result.stdout[-2000:])

## 10. Visualise Predictions

In [ ]:
pred_files = sorted(Path(TEST_OUTPUT).glob("*.nii.gz"))
print(f"Found {len(pred_files)} predictions")

PRED_IDX = 0   # change to view a different volume

pred_path = pred_files[PRED_IDX]
img_path  = Path(TEST_INPUT) / pred_path.name.replace(".nii.gz", "_0000.nii.gz")

pred_vol = nib.load(pred_path).get_fdata().astype(np.uint8).transpose(2, 1, 0)
ct_vol   = np.clip(nib.load(img_path).get_fdata().astype(np.float32).transpose(2, 1, 0), -150, 250)

tumor_counts = [(pred_vol[z] == 2).sum() for z in range(pred_vol.shape[0])]
show_slices  = sorted(range(len(tumor_counts)), key=lambda z: tumor_counts[z], reverse=True)[:4]

print(f"Volume      : {pred_path.name}")
print(f"Tumor slices: {sum(1 for t in tumor_counts if t > 0)} / {len(tumor_counts)}")

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for ax, t in zip(axes[0], ["CT", "Prediction", "Overlay"]):
    ax.set_title(t, fontsize=11, fontweight="bold")

for row, z in enumerate(show_slices):
    axes[row, 0].imshow(ct_vol[z],   cmap="gray")
    axes[row, 0].set_ylabel(f"Slice {z}\ntumor={tumor_counts[z]}px", fontsize=8)
    axes[row, 1].imshow(pred_vol[z], cmap="viridis", vmin=0, vmax=2)
    axes[row, 2].imshow(ct_vol[z],   cmap="gray")
    axes[row, 2].imshow(pred_vol[z], cmap="jet", alpha=0.4, vmin=0, vmax=2)
    for ax in axes[row]: ax.axis("off")

plt.suptitle(f"nnU-Net 2D — {pred_path.name}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"nnunet_pred_{pred_path.stem}.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Key File Locations

In [ ]:
model_dir = Path(NNUNET_RESULTS) / DATASET_NAME / f"{TRAINER}__nnUNetPlans__2d"

print("Checkpoints:")
for fold in range(5):
    fold_dir = model_dir / f"fold_{fold}"
    for ckpt in ["checkpoint_best.pth", "checkpoint_final.pth"]:
        p = fold_dir / ckpt
        print(f"  {'✓' if p.exists() else '–'}  fold_{fold}/{ckpt}")

print()
print("Other outputs:")
for label, path in [
    ("Auto plan",        Path(NNUNET_PREP) / DATASET_NAME / "nnUNetPlans.json"),
    ("Fingerprint",      Path(NNUNET_PREP) / DATASET_NAME / "dataset_fingerprint.json"),
    ("Test predictions", Path(TEST_OUTPUT)),
]:
    print(f"  {'✓' if Path(path).exists() else '–'}  {label}: {path}")